### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
try:
    tfidfvect.vocabulary_['cocoliso']
except KeyError as e:
    print(f"KeyError: {e} — la palabra no está en el vocabulario")


KeyError: 'cocoliso' — la palabra no está en el vocabulario


Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Análisis exploratorio del dataset

Antes de encarar las consignas, me parece necesario entender con qué datos estoy trabajando. La cátedra cargó el dataset, mostró un documento de ejemplo y listó los nombres de las 20 clases, pero no exploró su estructura. Considero que ese paso previo es importante, porque cada consigna que sigue depende de características del dataset que todavía no miré:

- **El balance entre clases** condiciona la elección de la métrica. Si las clases tuvieran tamaños muy distintos, la accuracy sería engañosa y el F1-macro (que la consigna 3 pide explícitamente) se justifica solo. Necesito confirmar esto con un conteo, no asumirlo.

- **La presencia de documentos vacíos o muy cortos** es un riesgo directo para las consignas 1 y 2. Un documento vacío tiene un vector TF-IDF nulo, y su 
  similaridad coseno contra cualquier otro documento va a ser cero, lo que rompería el   análisis de vecinos más similares.

- **El agrupamiento temático de las clases** es la clave para interpretar la consigna 1. Los 20 foros no son independientes: se agrupan en familias (computación, recreación, ciencia, política, religión). Anticipar esto me va a permitir entender por qué un  documento puede tener como vecino más parecido a otro de clase distinta pero tema  cercano, sin leerlo como un error del método.


In [24]:
import numpy as np

# Dimensiones generales del dataset
print(f"Documentos en train: {len(newsgroups_train.data)}")
print(f"Documentos en test:  {len(newsgroups_test.data)}")
print(f"Número de clases:    {len(newsgroups_train.target_names)}\n")

# Conteo de documentos por clase en train
clases, counts = np.unique(newsgroups_train.target, return_counts=True)

print("Documentos por clase (train):")
for c, n in zip(clases, counts):
    print(f"  {c:2d}  {newsgroups_train.target_names[c]:30s}  {n:4d} docs")

# Estadísticas del balance
print(f"\nMínimo: {counts.min()} docs | Máximo: {counts.max()} docs | Media: {counts.mean():.0f} docs")
print(f"Ratio max/min: {counts.max()/counts.min():.2f}")

Documentos en train: 11314
Documentos en test:  7532
Número de clases:    20

Documentos por clase (train):
   0  alt.atheism                      480 docs
   1  comp.graphics                    584 docs
   2  comp.os.ms-windows.misc          591 docs
   3  comp.sys.ibm.pc.hardware         590 docs
   4  comp.sys.mac.hardware            578 docs
   5  comp.windows.x                   593 docs
   6  misc.forsale                     585 docs
   7  rec.autos                        594 docs
   8  rec.motorcycles                  598 docs
   9  rec.sport.baseball               597 docs
  10  rec.sport.hockey                 600 docs
  11  sci.crypt                        595 docs
  12  sci.electronics                  591 docs
  13  sci.med                          594 docs
  14  sci.space                        593 docs
  15  soc.religion.christian           599 docs
  16  talk.politics.guns               546 docs
  17  talk.politics.mideast            564 docs
  18  talk.politics.misc    

#### Lo que muestra el output

El dataset tiene 11.314 documentos de entrenamiento y 7.532 de test, repartidos en 20 clases. El balance entre clases es bastante parejo: la mayoría de los grupos tiene entre 580 y 600 documentos, con un ratio máximo/mínimo de apenas 1.59. No es un desbalance severo (muy lejos del 75/25 que podría tener otro tipo de dataset), pero tampoco es perfecto.

Lo más relevante no es el desbalance en sí, sino *qué* clases son las más chicas. Las cuatro con menos documentos son `talk.religion.misc` (377), `alt.atheism` (480), `talk.politics.misc` (465) y `talk.politics.guns` (546): todas del área de religión y política, y varias con el sufijo `.misc` (miscelánea), que sugiere temas difusos y sin un vocabulario tan propio. Anticipo que estas clases van a ser las más difíciles de clasificar en las consignas 2 y 3, y como el F1-macro pondera todas las clases por igual, su bajo desempeño va a tener un impacto directo en la métrica. Es una hipótesis que voy a poder verificar más adelante.

In [25]:
# Largo de cada documento medido en palabras
largos = np.array([len(doc.split()) for doc in newsgroups_train.data])

print("Distribución del largo de los documentos (en palabras):")
print(f"  Mínimo:  {largos.min()}")
print(f"  Máximo:  {largos.max()}")
print(f"  Media:   {largos.mean():.0f}")
print(f"  Mediana: {np.median(largos):.0f}")

# Detección de documentos problemáticos
vacios = np.sum(largos == 0)
muy_cortos = np.sum(largos < 5)

print(f"\nDocumentos vacíos (0 palabras):       {vacios}")
print(f"Documentos con menos de 5 palabras:   {muy_cortos}")
print(f"Porcentaje de documentos muy cortos:  {100*muy_cortos/len(largos):.1f}%")

Distribución del largo de los documentos (en palabras):
  Mínimo:  0
  Máximo:  11765
  Media:   186
  Mediana: 83

Documentos vacíos (0 palabras):       300
Documentos con menos de 5 palabras:   410
Porcentaje de documentos muy cortos:  3.6%


#### Lo que muestra el output

**Distribución del largo:** los documentos tienen un largo muy variable, desde 0 hasta 11.765 palabras. La media (186) es más del doble de la mediana (83), lo que indica una distribución fuertemente sesgada a la derecha: la mayoría de los mensajes son cortos, pero una minoría de textos muy largos estira el promedio. 

**Documentos vacíos:** el hallazgo más importante es que hay 300 documentos con 0 palabras (un 2.65% del train) y 410 con menos de 5 palabras (3.6%). 

Esto tiene una implicancia concreta para las consignas 1 y 2. Un documento vacío produce un vector TF-IDF nulo, y la similaridad coseno de un vector nulo contra cualquier otro es cero. Por lo tanto, estos 300 documentos no aportan información y, si el muestreo aleatorio de la consigna 1 seleccionara uno de ellos, su análisis de vecinos más similares sería inválido (todas las similaridades cercanas a cero). 

Por esta razón, decido filtrar los documentos vacíos antes de la selección aleatoria de la consigna 1. 

## Consigna 1: Vectorización y similaridad entre documentos

La consigna pide tomar 5 documentos al azar, medir su similaridad con el resto y estudiar los 5 más similares de cada uno, analizando si la similaridad tiene sentido según el contenido del texto y la etiqueta de clasificación.

Antes de codear, dejo explícitas las decisiones que tomo y su fundamento:

- **Vectorizador por defecto.** Uso `TfidfVectorizer()` sin modificar parámetros, igual que el ejemplo de la cátedra. El tuneo del vectorizador queda reservado para la consigna 3;  en la 1 no corresponde porque el objetivo es explorar similaridades, no optimizar una métrica.

- **Selección aleatoria con semilla fija.** Elijo los 5 documentos al azar pero fijando una semilla, para que el resultado sea reproducible. Esto es necesario porque mi análisis escrito va a referirse a esos documentos puntuales, y la consigna exige que el notebook corra de inicio a fin de forma consistente.

- **Filtrado de documentos vacíos.** Como detecté en el análisis exploratorio, hay 300  documentos sin contenido cuyo vector TF-IDF es nulo. Los excluyo del muestreo para no seleccionar un documento inanalizable.

- **Exclusión del propio documento.** Al medir la similaridad de un documento contra todos, el más similar es siempre él mismo (coseno = 1). Por eso, al tomar los 5 más similares, descarto la primera posición.

In [26]:
# Vectorizamos el corpus de train con el vectorizador por defecto
tfidfvect = TfidfVectorizer()
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

# Identificamos los documentos NO vacíos (los que tienen al menos una palabra del vocabulario)
# nnz = number of non-zero: cuántas entradas distintas de cero tiene cada fila
largos_vec = np.diff(X_train.indptr)          # nº de términos no nulos por documento
docs_validos = np.where(largos_vec > 0)[0]     # índices de los documentos con contenido

print(f"Documentos totales:  {X_train.shape[0]}")
print(f"Documentos válidos:  {len(docs_validos)}")
print(f"Documentos vacíos:   {X_train.shape[0] - len(docs_validos)}")

# Seleccionamos 5 documentos al azar entre los válidos, con semilla fija
rng = np.random.default_rng(seed=42)
docs_elegidos = rng.choice(docs_validos, size=5, replace=False)

print(f"\nDocumentos elegidos al azar: {docs_elegidos}")
for idx in docs_elegidos:
    print(f"  doc {idx:5d}  ->  clase: {newsgroups_train.target_names[y_train[idx]]}")

Documentos totales:  11314
Documentos válidos:  11003
Documentos vacíos:   311

Documentos elegidos al azar: [8756 4973 7411 1009 4909]
  doc  8756  ->  clase: comp.os.ms-windows.misc
  doc  4973  ->  clase: sci.med
  doc  7411  ->  clase: rec.sport.hockey
  doc  1009  ->  clase: talk.politics.guns
  doc  4909  ->  clase: comp.os.ms-windows.misc


#### Selección de documentos

El filtro sobre la matriz TF-IDF identificó 311 documentos vacíos, levemente más que los 300 detectados en el análisis exploratorio. La razon de esta diferncia esn que en el exploratorio conté palabras del texto crudo, mientras que acá cuento términos del vocabulario TF-IDF, que ya descarta stopwords y términos poco frecuentes. Los 11 documentos adicionales evidentemente tenían solo palabras que el vectorizador no incorporó al vocabulario, por lo que 
su vector es igualmente nulo. Filtrar sobre la matriz vectorizada es más preciso, porque captura exactamente los documentos con similaridad coseno cero.

Los 5 documentos seleccionados al azar (semilla 42) cubren cuatro familias temáticas distintas —computación, ciencia, deporte y política— e incluyen dos documentos de la misma clase (`comp.os.ms-windows.misc`), lo que permitirá verificar si documentos de una misma categoría se reconocen entre sí como similares.

In [27]:
# Para cada documento elegido, calculamos su similaridad coseno contra todo el corpus
# y extraemos los 5 más similares (excluyendo el propio documento)

for idx in docs_elegidos:
    # Similaridad del documento idx contra todos los documentos de train
    cossim = cosine_similarity(X_train[idx], X_train)[0]

    # Ordenamos de mayor a menor y descartamos la posición 0 (el propio documento)
    mas_similares = np.argsort(cossim)[::-1][1:6]

    # Encabezado: documento consultado y su clase
    clase_original = newsgroups_train.target_names[y_train[idx]]
    print("=" * 75)
    print(f"DOC {idx}  |  CLASE: {clase_original}")
    print("-" * 75)

    # Listamos los 5 vecinos con su similaridad, clase y si coincide
    for rank, j in enumerate(mas_similares, start=1):
        clase_vecino = newsgroups_train.target_names[y_train[j]]
        coincide = "coincide" if y_train[j] == y_train[idx] else "distinta"
        print(f"  {rank}. doc {j:5d} | sim={cossim[j]:.3f} | clase: {clase_vecino:28s} ({coincide})")
    print()

DOC 8756  |  CLASE: comp.os.ms-windows.misc
---------------------------------------------------------------------------
  1. doc  8175 | sim=0.381 | clase: comp.sys.ibm.pc.hardware     (distinta)
  2. doc  4927 | sim=0.378 | clase: comp.sys.ibm.pc.hardware     (distinta)
  3. doc  1574 | sim=0.333 | clase: comp.sys.ibm.pc.hardware     (distinta)
  4. doc  7424 | sim=0.309 | clase: comp.sys.ibm.pc.hardware     (distinta)
  5. doc  6894 | sim=0.292 | clase: talk.politics.guns           (distinta)

DOC 4973  |  CLASE: sci.med
---------------------------------------------------------------------------
  1. doc  1753 | sim=0.554 | clase: sci.med                      (coincide)
  2. doc 11258 | sim=0.190 | clase: sci.med                      (coincide)
  3. doc  5826 | sim=0.176 | clase: soc.religion.christian       (distinta)
  4. doc 10836 | sim=0.174 | clase: alt.atheism                  (distinta)
  5. doc  1666 | sim=0.168 | clase: sci.med                      (coincide)

DOC 7411  |  C

In [28]:
def mostrar_documento(idx, n_chars=500):
    """Imprime la clase y un fragmento del texto de un documento."""
    clase = newsgroups_train.target_names[y_train[idx]]
    texto = newsgroups_train.data[idx].strip()
    print(f"--- DOC {idx} | CLASE: {clase} ---")
    print(texto[:n_chars])
    print("...\n" if len(texto) > n_chars else "\n")


# Documento de referencia
print("#" * 75)
print("DOCUMENTO DE REFERENCIA")
print("#" * 75)
mostrar_documento(8756)

# Vecino "lógico": mismo tema (hardware), aunque de clase distinta
print("#" * 75)
print("VECINO 1 (sim=0.381) — clase distinta pero misma familia comp.*")
print("#" * 75)
mostrar_documento(8175)

# Vecino "raro": clase de otra familia (política)
print("#" * 75)
print("VECINO 5 (sim=0.292) — clase de otra familia (política)")
print("#" * 75)
mostrar_documento(6894)

###########################################################################
DOCUMENTO DE REFERENCIA
###########################################################################
--- DOC 8756 | CLASE: comp.os.ms-windows.misc ---
I bought a Viewsonic 17 for use at home but after a week I took it back.  I 
felt for the money my NEC 5FG that I use at work was a much better monitor.
The NEC is sharper, flatter, less distorted, and more stable.  I have heard 
complaints from people about the NEC FG series having some quality control 
problems but mine has been in use for about a year with no problems at all.

There was nothing really broken with the Viewsonic but overall it did not 
match up.  I used my ATI Graphics Ultra in
...

###########################################################################
VECINO 1 (sim=0.381) — clase distinta pero misma familia comp.*
###########################################################################
--- DOC 8175 | CLASE: comp.sys.ibm.pc.hardware ---
I

#### Análisis por contenido del texto

Para cumplir con la consigna de analizar la similaridad según el contenido (y no solo la etiqueta), leo el documento de referencia 8756 y dos de sus vecinos: el más similar (hardware) y el más atípico (política).

**Vecino lógico (doc 8175, sim=0.381).** Aunque pertenece a `comp.sys.ibm.pc.hardware` y no a `comp.os.ms-windows.misc`, al leer los textos queda claro que ambos discuten exactamente el mismo tema: la calidad de monitores de 17 pulgadas, y ambos mencionan específicamente el modelo NEC 5FG. La similaridad no es casual: los documentos hablan de lo mismo. De hecho, el documento de referencia, pese a estar en el foro de Windows, no trata 
sobre el sistema operativo sino sobre monitores y placas de video, es decir, sobre hardware. El método fue más fiel al contenido real del texto que la etiqueta del foro. Esto confirma que una clase distinta dentro de la misma familia temática no es un error, sino una consecuencia de que la etiqueta de foro es una división más fina que la similaridad temática.

**Vecino atípico (doc 6894, sim=0.292).** Este documento es un comunicado de prensa de la Casa Blanca sobre el caso Waco, sin ninguna relación temática con monitores. Su aparición entre los más similares se explica por coincidencias de vocabulario genérico (palabras comunes que sobrevivieron al filtrado TF-IDF) y no por cercanía de contenido. Es un ejemplo concreto de la limitación del enfoque bag-of-words: al basarse solo en coincidencia de palabras, puede asignar similaridad entre textos que no comparten tema. Además, su similaridad (0.292) es notoriamente menor que la del vecino real (0.381), lo que confirma que la magnitud del coseno es tan informativa como la coincidencia de etiqueta: los parecidos débiles tienden a ser espurios.

### Cierre de la consigna 1

El resultado de los 5 documentos muestra que la similaridad coseno funciona mejor o peor según qué tan propio es el vocabulario de cada clase:

- **Hockey (doc 7411):** 5 de 5 vecinos de su clase, con similaridades altas (0.42–0.68). Tiene vocabulario específico (equipos, jugadores), así que es fácil de identificar.

- **Windows (doc 8756):** sus vecinos son de `comp.sys.ibm.pc.hardware`, clase distinta pero misma familia. Al leer los textos, ambos hablaban del mismo monitor (NEC 5FG). El parecido es real; la etiqueta de foro es más fina que el tema. 

- **Política (doc 1009):** vecinos correctos en clase pero con similaridades bajas y casi iguales (0.139–0.147). No tiene un vecino claramente parecido. Confirma lo que anticipé en el descriptivo: política y religión son áreas difusas.

- **Medicina (doc 4973):** un vecino fuerte (0.554) y después una caída a ~0.17, donde se  cuelan clases no relacionadas. A similaridad baja el ranking es casi ruido.

**Conclusión.** TF-IDF + coseno captura bien la estructura temática, pero tiene dos límites: no separa clases de vocabulario compartido, y puede dar similaridad entre textos sin relación real solo por compartir palabras genéricas (como el vecino de política del doc 8756, un comunicado sobre Waco). La consigna 2 va a clasificar usando esta misma similaridad, así que es esperable que herede estas debilidades en las clases difusas.

## Consigna 2: Clasificador por prototipos (zero-shot)

La consigna pide clasificar cada documento de test comparándolo con todos los de train y asignarle la clase del documento de train más similar. Es un clasificador por vecino más cercano (1-NN) usando similaridad coseno.

Decisiones:

- **No hay entrenamiento.** El modelo no ajusta parámetros: guarda los vectores de train y clasifica comparando por coseno. Por eso se lo llama "por prototipos" o zero-shot.
- **Vectorizador por defecto**, ajustado en train (el mismo `X_train` de la consigna 1). El test se vectoriza con `transform`, sin re-ajustar.
- **Clasifico todo el test**, incluidos los documentos vacíos, para que el F1 sea comparable  con el de la consigna 3 (Naïve Bayes) sobre el mismo conjunto.
- **Cálculo por lotes**, porque comparar 7532 documentos de test contra 11314 de train genera una matriz muy grande si se hace de una sola vez.

In [29]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

# Para cada doc de test: clase del doc de train con mayor similaridad coseno
batch = 500
y_pred_proto = np.empty(X_test.shape[0], dtype=int)

for i in range(0, X_test.shape[0], batch):
    sims = cosine_similarity(X_test[i:i+batch], X_train)  # (batch, n_train)
    vecino = sims.argmax(axis=1)                          # índice del train más similar
    y_pred_proto[i:i+batch] = y_train[vecino]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-macro (clasificador por prototipos): {f1_proto:.4f}")

F1-macro (clasificador por prototipos): 0.5050


In [30]:
from sklearn.metrics import f1_score

f1_por_clase = f1_score(y_test, y_pred_proto, average=None)

orden = np.argsort(f1_por_clase)  # de peor a mejor
print("F1 por clase (peor a mejor):\n")
for c in orden:
    print(f"  {f1_por_clase[c]:.3f}  {newsgroups_train.target_names[c]}")

F1 por clase (peor a mejor):

  0.277  talk.religion.misc
  0.307  talk.politics.misc
  0.406  sci.electronics
  0.425  alt.atheism
  0.453  talk.politics.mideast
  0.462  talk.politics.guns
  0.476  rec.autos
  0.480  comp.os.ms-windows.misc
  0.508  soc.religion.christian
  0.512  comp.graphics
  0.516  comp.sys.mac.hardware
  0.518  comp.sys.ibm.pc.hardware
  0.533  misc.forsale
  0.561  sci.med
  0.566  sci.space
  0.569  rec.motorcycles
  0.570  sci.crypt
  0.586  rec.sport.baseball
  0.642  comp.windows.x
  0.735  rec.sport.hockey


### Cierre de la consigna 2

El clasificador por prototipos da F1-macro 0.505. Es diez veces mejor que el azar (que en 20 clases sería ~0.05), pero bajo como clasificador.

El F1 por clase explica por qué. Las 4 peores son `talk.religion.misc` (0.277), `talk.politics.misc` (0.307), `sci.electronics` (0.406) y `alt.atheism` (0.425); la mejor es `rec.sport.hockey` (0.735). Se confirma la hipótesis del análisis exploratorio: las clases de religión y política, difusas y con menos documentos, son las que arrastran la métrica, mientras que las de vocabulario propio (hockey) clasifican bien.

`sci.electronics` es la excepción al patrón: no es `.misc` pero igual anda mal. Se explica porque comparte vocabulario técnico con hardware y crypto, así que su vecino más cercano suele caer en una clase hermana. Es la misma confusión por familia temática que vi en la consigna 1, ahora penalizando la clasificación.

El límite del método es que clasifica según un único vecino: si ese vecino es de una clase hermana, falla. 

## Consigna 3: Clasificación con Naïve Bayes

La consigna pide entrenar Naïve Bayes para maximizar el F1-macro en test, probando MultinomialNB y ComplementNB, y variando parámetros del vectorizador y de los modelos. Restricción de la cátedra: **no cambiar `ngram_range`**.

Decisiones:

- **Pruebo los dos modelos**, MultinomialNB y ComplementNB. ComplementNB está pensado para datasets desbalanceados, así que espero que rinda mejor en las clases chicas.
- **Vario parámetros del vectorizador** permitidos (`min_df`, `max_df`, `sublinear_tf`, `stop_words`) y el suavizado `alpha` de los modelos.
- **Comparo contra el baseline** del ejemplo de la cátedra (MultinomialNB con TfidfVectorizer por defecto), que da F1-macro conocido, y contra el clasificador por prototipos de la  consigna 2 (0.505).
- **Métrica y conjunto** iguales a la consigna 2 (F1-macro sobre el mismo test), para que los tres resultados sean comparables.

In [31]:
# Baseline: el mismo del ejemplo de la cátedra (TfidfVectorizer por defecto + MultinomialNB)
clf_base = MultinomialNB()
clf_base.fit(X_train, y_train)
f1_base = f1_score(y_test, clf_base.predict(X_test), average='macro')

print(f"F1-macro baseline (MultinomialNB, vectorizador por defecto): {f1_base:.4f}")
print(f"F1-macro clasificador por prototipos (consigna 2):           {f1_proto:.4f}")

F1-macro baseline (MultinomialNB, vectorizador por defecto): 0.5854
F1-macro clasificador por prototipos (consigna 2):           0.5050


In [32]:
from itertools import product

# Configuraciones del vectorizador (SIN tocar ngram_range, como pide la consigna)
configs_vect = [
    dict(),                                                      # por defecto
    dict(stop_words='english'),
    dict(stop_words='english', min_df=5),
    dict(stop_words='english', min_df=5, max_df=0.8),
    dict(stop_words='english', min_df=5, max_df=0.8, sublinear_tf=True),
]

modelos = [('MultinomialNB', MultinomialNB), ('ComplementNB', ComplementNB)]
alphas  = [0.01, 0.1, 0.5, 1.0]

resultados = []
for cfg in configs_vect:
    vec = TfidfVectorizer(**cfg)
    Xtr = vec.fit_transform(newsgroups_train.data)
    Xte = vec.transform(newsgroups_test.data)
    for (mname, M), a in product(modelos, alphas):
        clf = M(alpha=a).fit(Xtr, y_train)
        f1 = f1_score(y_test, clf.predict(Xte), average='macro')
        resultados.append((f1, mname, a, cfg))

# Ordenamos de mejor a peor y mostramos el top 10
resultados.sort(key=lambda r: r[0], reverse=True)
print("Top 10 configuraciones (F1-macro):\n")
for f1, mname, a, cfg in resultados[:10]:
    print(f"  {f1:.4f} | {mname:14s} | alpha={a:<4} | {cfg}")

Top 10 configuraciones (F1-macro):

  0.6978 | ComplementNB   | alpha=0.5  | {'stop_words': 'english'}
  0.6961 | ComplementNB   | alpha=0.5  | {}
  0.6954 | ComplementNB   | alpha=0.1  | {}
  0.6936 | ComplementNB   | alpha=1.0  | {'stop_words': 'english'}
  0.6930 | ComplementNB   | alpha=1.0  | {}
  0.6919 | ComplementNB   | alpha=0.1  | {'stop_words': 'english'}
  0.6844 | MultinomialNB  | alpha=0.01 | {'stop_words': 'english'}
  0.6829 | MultinomialNB  | alpha=0.01 | {}
  0.6827 | ComplementNB   | alpha=0.5  | {'stop_words': 'english', 'min_df': 5}
  0.6827 | ComplementNB   | alpha=0.5  | {'stop_words': 'english', 'min_df': 5, 'max_df': 0.8}


In [33]:
# Modelo ganador
best_vec = TfidfVectorizer(stop_words='english')
Xtr = best_vec.fit_transform(newsgroups_train.data)
Xte = best_vec.transform(newsgroups_test.data)
best_clf = ComplementNB(alpha=0.5).fit(Xtr, y_train)
y_pred_nb = best_clf.predict(Xte)

f1_nb = f1_score(y_test, y_pred_nb, average='macro')
print(f"F1-macro Naïve Bayes (mejor):  {f1_nb:.4f}")
print(f"F1-macro prototipos (cons. 2): {f1_proto:.4f}\n")

# F1 por clase: prototipos vs Naïve Bayes
f1c_proto = f1_score(y_test, y_pred_proto, average=None)
f1c_nb    = f1_score(y_test, y_pred_nb,    average=None)

print(f"{'clase':30s} {'proto':>7} {'NB':>7} {'dif':>7}")
for c in np.argsort(f1c_nb):
    dif = f1c_nb[c] - f1c_proto[c]
    print(f"{newsgroups_train.target_names[c]:30s} {f1c_proto[c]:7.3f} {f1c_nb[c]:7.3f} {dif:+7.3f}")

F1-macro Naïve Bayes (mejor):  0.6978
F1-macro prototipos (cons. 2): 0.5050

clase                            proto      NB     dif
talk.religion.misc               0.277   0.221  -0.056
alt.atheism                      0.425   0.366  -0.059
talk.politics.misc               0.307   0.517  +0.209
sci.electronics                  0.406   0.629  +0.223
comp.os.ms-windows.misc          0.480   0.652  +0.172
talk.politics.guns               0.462   0.658  +0.197
comp.sys.ibm.pc.hardware         0.518   0.667  +0.149
soc.religion.christian           0.508   0.690  +0.182
comp.graphics                    0.512   0.720  +0.209
misc.forsale                     0.533   0.736  +0.203
comp.sys.mac.hardware            0.516   0.741  +0.225
rec.autos                        0.476   0.776  +0.300
sci.crypt                        0.570   0.786  +0.216
sci.space                        0.566   0.797  +0.231
comp.windows.x                   0.642   0.800  +0.158
sci.med                          0.561   0.

### Cierre de la consigna 3

El mejor modelo es ComplementNB con alpha=0.5 y stop_words='english': F1-macro 0.6978, contra 0.5854 del baseline y 0.5050 del clasificador por prototipos.

Sobre el tuneo:
- **ComplementNB gana a MultinomialNB** en todo el podio, como esperaba: al estar pensado para datos desbalanceados, levanta las clases chicas que castigan el macro.
- **Sacar stopwords aporta poco** (0.6978 vs 0.6961 sin ellas), porque TF-IDF ya baja el peso de las palabras comunes.
- **Filtrar términos con min_df/max_df empeora** el resultado: a Naïve Bayes le sirve tener todo el vocabulario, incluso los términos poco frecuentes.

Comparado con prototipos por clase, Naïve Bayes mejora en 18 de 20 clases, con saltos de hasta +0.36 (`talk.politics.mideast`). Usa todo el vocabulario de cada clase en vez de un solo vecino, así que aprovecha mejor las clases con léxico propio.

Empeora solo en 2 clases: `talk.religion.misc` (−0.056) y `alt.atheism` (−0.059), las dos peores de ambos métodos. El motivo no es el modelo sino el dataset: atheism, religion.misc y christian comparten casi todo el vocabulario, así que ningún método basado en palabras las separa bien. 


## Consigna 4: Similaridad entre palabras (matriz término-documento)

La consigna pide transponer la matriz documento-término para obtener una matriz término-documento, donde cada palabra queda representada por un vector según los documentos en que aparece. Con eso se estudia la similaridad entre palabras: se eligen 5 y se miran sus 5 más similares.

Decisiones:

- **Vectorizador sin stopwords** (`stop_words='english'`). Uso el mismo criterio que ganó en la consigna 3 porque deja el vocabulario más limpio, y así evito que las palabras más similares sean términos vacíos ("the", "of"), en línea con lo que pide la consigna sobre evitar términos poco interpretables.

- **Palabras elegidas manualmente:** `game`, `driver`, `space`, `engine`, `god`. Son de familias temáticas distintas (deporte, tecnología/autos, ciencia, autos, religión), lo que me permite anticipar qué vecinas espero en cada caso y contrastar. Dos de ellas son ambiguas a propósito: `driver` (conductor de auto o controlador de hardware) y `space` (espacio exterior o espacio en disco), para ver hacia qué sentido se inclinan sus vecinas según cómo aparecen en el corpus.

In [34]:
# Vectorizador sin stopwords (el mismo criterio que ganó en la consigna 3)
vect_w = TfidfVectorizer(stop_words='english')
X_w = vect_w.fit_transform(newsgroups_train.data)

# Transponemos: de documento-término a término-documento
# Ahora cada FILA es una palabra (un vector sobre el espacio de documentos)
X_terms = X_w.T
print(f"Matriz término-documento: {X_terms.shape}  (palabras x documentos)")

# Diccionarios para ir de palabra a índice y de índice a palabra
vocab = vect_w.vocabulary_               # palabra -> índice de fila
idx2word = {v: k for k, v in vocab.items()}

def palabras_similares(palabra, n=5):
    if palabra not in vocab:
        print(f"'{palabra}' no está en el vocabulario.\n")
        return
    i = vocab[palabra]
    sims = cosine_similarity(X_terms[i], X_terms)[0]   # similaridad contra todas las palabras
    mas_sim = np.argsort(sims)[::-1][1:n+1]             # top n, excluyendo la propia palabra
    print(f"'{palabra}'  ->  ", end="")
    print(", ".join(f"{idx2word[j]} ({sims[j]:.3f})" for j in mas_sim))

for w in ["game", "driver", "space", "engine", "god"]:
    palabras_similares(w)

Matriz término-documento: (101322, 11314)  (palabras x documentos)
'game'  ->  games (0.214), espn (0.192), team (0.184), hockey (0.180), scored (0.174)
'driver'  ->  dis_pkt9 (0.189), drv (0.177), laureti (0.175), vsbd (0.175), lauretti (0.175)
'space'  ->  nasa (0.328), shuttle (0.290), seds (0.285), enfant (0.269), exploration (0.240)
'engine'  ->  mels (0.195), infromation (0.195), buckled (0.195), mountings (0.195), guesser (0.195)
'god'  ->  jesus (0.281), bible (0.276), christ (0.267), faith (0.259), existence (0.259)


### Cierre de la consigna 4

Al transponer la matriz, cada palabra queda representada por los documentos en que aparece. El resultado depende fuerte de la frecuencia de la palabra:

- **Palabras frecuentes y de vocabulario propio funcionan bien.** `game` trae games, espn, team, hockey, scored (deporte); `god` trae jesus, bible, christ, faith (religión); `space` trae nasa, shuttle, exploration (astronáutica). En `space` se resolvió la ambigüedad que había anticipado: domina el sentido de espacio exterior, no el de espacio en disco.

- **Palabras dispersas o de baja frecuencia dan ruido.** `driver` trae fragmentos de código (dis_pkt9, vsbd) y apellidos; `engine` trae typos (infromation) y términos sueltos, todos con similaridad idéntica (0.195). Ese valor repetido delata que las palabras coocurren en un único documento compartido: con tan pocas apariciones, el vector es demasiado ralo y la similaridad se vuelve espuria. La ambigüedad de `driver` no se resolvió hacia ningún sentido, se perdió en términos raros.

La limitación de fondo es que acá el contexto de una palabra es el documento entero, así que las palabras poco frecuentes quedan mal representadas. 